# Evaluation

Run this after `04_full_training.ipynb`. It reloads the saved model and evaluates it using the same `L.ipynb` dataset and sorted-folder label map.

In [ ]:
from pathlib import Path
import ast
import json
import os
import random
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

# mount Google Drive when running inside Colab.
if Path('/content').exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Google Drive mount skipped:', exc)

# Local fallback path: <repo>/data/mini_dataset, then <repo>/data.
COLAB_DATASET_DIR = Path('/content/drive/MyDrive/lipy/mini_dataset')
LOCAL_MINI_DATASET_DIR = PROJECT_ROOT / 'data' / 'mini_dataset'
LOCAL_DATA_DIR = PROJECT_ROOT / 'data'

if COLAB_DATASET_DIR.exists():
    DATASET_DIR = COLAB_DATASET_DIR
elif LOCAL_MINI_DATASET_DIR.exists():
    DATASET_DIR = LOCAL_MINI_DATASET_DIR
else:
    DATASET_DIR = LOCAL_DATA_DIR

MIN_IMAGES = 25
IMG_SIZE = 64
BATCH_SIZE = 32
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'training'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ARTIFACT_DIR = PROJECT_ROOT / 'outputs' / 'models'
METRICS_DIR = PROJECT_ROOT / 'outputs' / 'metrics'
MODEL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_MODEL_ARTIFACT_DIR = Path('/content/drive/MyDrive/lipy_models')
if Path('/content/drive/MyDrive').exists():
    DRIVE_MODEL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
else:
    DRIVE_MODEL_ARTIFACT_DIR = None

RUN_ID = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
DEPLOY_MODEL_PATH = PROJECT_ROOT / 'backend' / 'models' / 'odia_ocr_cnn.keras'

def model_artifact_path(model_family):
    safe_family = str(model_family).strip().lower().replace(' ', '_')
    return MODEL_ARTIFACT_DIR / f'lipy_odia_ocr_{safe_family}_{RUN_ID}.keras'


def save_model_to_local_and_drive(model, model_family):
    local_path = model_artifact_path(model_family)
    model.save(local_path)
    print('Saved local model:', local_path)

    drive_path = None
    if DRIVE_MODEL_ARTIFACT_DIR is not None:
        drive_path = DRIVE_MODEL_ARTIFACT_DIR / local_path.name
        model.save(drive_path)
        print('Saved Google Drive model:', drive_path)
    else:
        print('Google Drive model save skipped: Drive is not mounted/available.')

    return local_path, drive_path


def latest_local_model():
    models = sorted(MODEL_ARTIFACT_DIR.glob('lipy_odia_ocr_*.keras'), key=lambda p: p.stat().st_mtime, reverse=True)
    if models:
        return models[0]
    if DEPLOY_MODEL_PATH.exists():
        return DEPLOY_MODEL_PATH
    raise FileNotFoundError('No model found in outputs/models or backend/models.')

print('Project Root:', PROJECT_ROOT)
print('Dataset Path:', DATASET_DIR)
print('TensorFlow:', tf.__version__)
print('GPU Devices:', tf.config.list_physical_devices('GPU'))

In [ ]:
def list_class_folders(dataset_dir):
    if not dataset_dir.exists():
        raise FileNotFoundError(f'Dataset folder does not exist: {dataset_dir}')
    return [p.name for p in dataset_dir.iterdir() if p.is_dir()]


def count_images(class_folder):
    folder_path = DATASET_DIR / class_folder
    return len([p for p in folder_path.iterdir() if p.is_file()])

# list all folders/classes.
classes = list_class_folders(DATASET_DIR)
print('Total Classes:', len(classes))
print(classes[:10])

# keep classes with enough images.
valid_classes = []
for folder in classes:
    image_count = count_images(folder)
    if image_count >= MIN_IMAGES:
        valid_classes.append(folder)
        print(f'{folder} -> {image_count} images')

# sorting gives consistent label ordering.
valid_classes.sort()
print('Valid Classes:', len(valid_classes))
print(valid_classes)

# class-name to integer label.
label_map = {class_name: idx for idx, class_name in enumerate(valid_classes)}
reverse_label_map = {idx: class_name for class_name, idx in label_map.items()}
num_classes = len(valid_classes)
print('Number of Classes:', num_classes)
print(label_map)

if num_classes < 2:
    raise ValueError('Need at least two valid class folders. Check DATASET_DIR and MIN_IMAGES.')

(OUTPUT_DIR / 'label_map.json').write_text(json.dumps(label_map, indent=2, ensure_ascii=False), encoding='utf-8')

In [ ]:
images = []
labels = []
loaded_count = 0
failed_count = 0

for class_name in valid_classes:
    class_path = DATASET_DIR / class_name
    label = label_map[class_name]

    for image_name in os.listdir(class_path):
        img_path = class_path / image_name
        if not img_path.is_file():
            continue

        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            failed_count += 1
            print('Failed:', img_path)
            continue

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        img = img.astype('float32') / 255.0

        images.append(img)
        labels.append(label)
        loaded_count += 1

print('Images Loaded :', loaded_count)
print('Images Failed :', failed_count)

X = np.array(images, dtype='float32').reshape(-1, IMG_SIZE, IMG_SIZE, 1)
y_raw = np.array(labels, dtype='int32')
y = to_categorical(y_raw, num_classes=num_classes)

print('X Shape:', X.shape)
print('y Shape:', y.shape)

In [ ]:
X_train, X_test, y_train, y_test, y_train_raw, y_test_raw = train_test_split(
    X,
    y,
    y_raw,
    test_size=0.2,
    random_state=SEED,
    stratify=y_raw,
)

print('Training Shape:', X_train.shape)
print('Testing Shape :', X_test.shape)
print('Training Labels:', y_train.shape)
print('Testing Labels :', y_test.shape)

In [ ]:
def load_backend_class_names():
    labels_py = PROJECT_ROOT / 'backend' / 'labels.py'
    if not labels_py.exists():
        return []
    tree = ast.parse(labels_py.read_text(encoding='utf-8'))
    for node in tree.body:
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id == 'odia_ml_labels':
                    odia_ml_labels = ast.literal_eval(node.value)
                    return list(odia_ml_labels.values())
    return []

backend_class_names = load_backend_class_names()
if backend_class_names:
    if backend_class_names[:num_classes] == valid_classes and len(backend_class_names) == num_classes:
        print('Backend label order matches this training label_map.')
    else:
        print('Warning: backend/labels.py order differs from the L.ipynb sorted folder label_map.')
        print('Train/evaluate with this notebook label_map.json. For deployment, keep backend labels in the same order as label_map.json.')

In [ ]:
EVALUATION_MODEL_PATH = latest_local_model()
model = keras.models.load_model(EVALUATION_MODEL_PATH, compile=False)
print('Loaded model:', EVALUATION_MODEL_PATH)


In [ ]:
y_pred_prob = model.predict(X_test, verbose=0)
y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(y_pred_prob, axis=1)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print('Accuracy :', accuracy)
print('Precision:', precision)
print('Recall   :', recall)
print('F1 Score :', f1)

report = classification_report(y_true, y_pred, target_names=valid_classes, zero_division=0)
print('\nClassification Report')
print(report)
(OUTPUT_DIR / 'classification_report.txt').write_text(report, encoding='utf-8')

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=valid_classes, columns=valid_classes)
cm_df.to_csv(OUTPUT_DIR / 'confusion_matrix.csv')

plt.figure(figsize=(max(12, num_classes * 0.35), max(10, num_classes * 0.32)))
sns.heatmap(cm_df, annot=num_classes <= 30, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=220)
plt.show()

In [ ]:
idx = random.randint(0, len(X_test) - 1)
sample = X_test[idx]
prediction = model.predict(sample.reshape(1, IMG_SIZE, IMG_SIZE, 1), verbose=0)
predicted_class = int(np.argmax(prediction))
actual_class = int(np.argmax(y_test[idx]))

predicted_label = reverse_label_map[predicted_class]
actual_label = reverse_label_map[actual_class]

plt.imshow(sample.reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
plt.title(f'Predicted: {predicted_label}\nActual: {actual_label}')
plt.axis('off')
plt.show()